# Previsão da Arrecadação de Emolumentos Extrajudiciais do TJMA
### Projeto de Qualificação de Mestrado — PECS/UEMA

Notebook de demonstração para a banca examinadora: clona o repositório público, **exibe o código-fonte completo de cada módulo** (com destaque de sintaxe, exatamente como está no repositório — nenhuma cópia paralela) e roda a pipeline experimental de ponta a ponta (SARIMA/SARIMAX, LSTM em NumPy puro e XGBoost) sobre a série real de arrecadação do FERJ/TJMA (93 meses, jan/2019-set/2026).

Repositório: `github.com/ferdinan2016-pixel/tjma-emolumentos-forecast-demo-2` — público, sem necessidade de token.

## 1. Clonar o repositório e instalar dependências

In [ ]:
GITHUB_USER = "ferdinan2016-pixel"
REPO = "tjma-emolumentos-forecast-demo-2"

!rm -rf /content/{REPO}
!git clone --quiet https://github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!pip install -q -r requirements.txt

## 2. Código-fonte completo

As células abaixo exibem o conteúdo real dos arquivos recém-clonados (`Code(filename=...)`), não uma cópia colada no notebook — qualquer alteração no repositório aparece aqui na próxima execução.

In [ ]:
from IPython.display import Code, display, Markdown

def show(path, title):
    display(Markdown(f"#### `{path}`  —  {title}"))
    display(Code(filename=path, language="python"))

### 2.1 Dados: carga e pré-processamento

In [ ]:
show("src/data/real_data_loader.py", "carga da série real do FERJ (93 meses)")

In [ ]:
show("src/data/synthetic_generator.py", "série sintética de referência/placeholder metodológico")

In [ ]:
show("src/data/preprocessing.py", "split temporal, features de lag, janelas, normalização")

### 2.2 Modelos: as três arquiteturas comparadas

In [ ]:
show("src/models/classical.py", "Modelo 1 — SARIMA/SARIMAX (walk-forward e multi-step)")

In [ ]:
show("src/models/lstm_numpy.py", "Modelo 2 — LSTM implementada em NumPy puro (forward, BPTT, Adam)")

In [ ]:
show("src/models/xgboost_model.py", "Modelo 3 — XGBoost sobre features de defasagem")

### 2.3 Avaliação e visualização

In [ ]:
show("src/evaluation/metrics.py", "MAE, RMSE, MAPE")

In [ ]:
show("src/visualization/plots.py", "geração dos 4 gráficos comparativos")

### 2.4 Orquestração da pipeline

In [ ]:
show("scripts/run_pipeline.py", "ponto de entrada único: carga -> pré-processamento -> treino -> avaliação -> previsão -> gráficos")

## 3. Executando a pipeline completa

Treina as três arquiteturas sobre os 93 meses reais, avalia no período de teste (12 meses mais recentes) e gera a previsão multi-step dos 12 meses seguintes. Leva cerca de 15 segundos em CPU comum — nenhuma das três arquiteturas usa GPU nesta fase (ver discussão na Seção 2.5 abaixo).

In [ ]:
!python scripts/run_pipeline.py

## 4. Resultados

In [ ]:
import pandas as pd

print("Métricas no conjunto de teste:")
display(pd.read_csv("outputs/results/metricas_teste.csv", index_col=0))

print("\nPrevisão para os próximos 12 meses:")
display(pd.read_csv("outputs/results/previsao_proximos_12_meses.csv", index_col=0))

In [ ]:
from IPython.display import Image
import glob

for path in sorted(glob.glob("outputs/figures/*.png")):
    if "sintetico" not in path:
        print(path)
        display(Image(filename=path))

## 5. Sobre o uso de CPU em vez de GPU

A LSTM desta fase foi implementada em NumPy puro — de propósito, para manter o ambiente leve e sem dependência de CUDA, dado o volume ainda moderado de dados (93 meses). Por isso o pipeline inteiro roda em segundos numa CPU comum, e a GPU do Colab não é utilizada aqui. Para a fase de dissertação, o passo planejado é reimplementar `src/models/lstm_numpy.py` em PyTorch ou TensorFlow/Keras, o que viabiliza regularização (dropout), empilhamento de camadas e, aí sim, aceleração real por GPU — ver Seção 3.3.2 do Projeto de Qualificação.